In [40]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
from sklearn.datasets import make_classification
import torch

In [41]:
torch.manual_seed(42)

In [42]:
df = pd.read_csv("/content/fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [43]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [45]:
X_train = X_train/255.0
X_test = X_test/255.0

In [46]:
from torch.utils.data import Dataset, DataLoader

In [47]:
#create a CustomDataset class
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype = torch.long)
  def __len__(self):
    return self.features.shape[0]
  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [48]:
#create train_dataset object
train_dataset = CustomDataset(X_train,y_train)

In [49]:
#create test_dataset object
test_dataset = CustomDataset(X_test,y_test)


In [50]:
#create train and test loader
train_loader = DataLoader(train_dataset, batch_size = 32 ,shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle =False) # Changed train_dataset to test_dataset

The confusion here comes from mixing up **spatial dimensions** (height and width) with **channel dimensions** (the number of filters).

Here is the breakdown of why `BatchNorm2d` takes `32`:

**1. What `padding='same'` actually does**
Setting `padding='same'` ensures that the **height and width** of your input remain exactly the same after the convolution. For example, if you input a $64 \times 64$ image, the output will still be $64 \times 64$ spatially.

**2. What the `32` in `Conv2d` does**
The `32` in `nn.Conv2d(input_features, 32, ...)` dictates the **number of output channels** (the depth). It tells the network to apply 32 distinct $3 \times 3$ filters to the input. Even though the height and width stay the same, the *depth* of your tensor changes from `input_features` to `32`.

**3. Why `BatchNorm2d` needs `32**`
`nn.BatchNorm2d` applies normalization *per channel*. It needs to know exactly how many channels are coming in so it can assign a specific mean, variance, learned scaling weight ($\gamma$), and shift bias ($\beta$) to each individual channel.

Here is how the shape of your tensor transforms through this block (assuming a batch size of $B$ and a spatial size of $H \times W$):

* **Input shape:** $(B, \text{input\_features}, H, W)$
* **After `Conv2d`:** $(B, \mathbf{32}, H, W)$ *(H and W stayed the same due to padding, but channels became 32)*
* **After `ReLU`:** $(B, \mathbf{32}, H, W)$
* **After `BatchNorm2d(32)`:** $(B, \mathbf{32}, H, W)$ *(It expects the 32 channels created by the Conv2d layer)*
* **After `MaxPool2d`:** $(B, 32, H/2, W/2)$ *(Spatial dimensions are halved, channels remain 32)*

In short: `padding='same'` keeps the Height and Width the same, but the Conv2d layer still changes the output Channels to 32. Because `BatchNorm2d` normalizes across channels, it must match that new channel count of 32.

In [51]:
from torch.nn.modules.batchnorm import BatchNorm1d
from torch.nn.modules.activation import ReLU
#define NN class
class MyNN(nn.Module):
  def __init__(self,input_features):

    super().__init__()
    self.features  =nn.Sequential(
        nn.Conv2d(input_features,32,kernel_size = 3,padding = 'same'),#input features is basically channel number
        #32 is basically the number of filters, out channel , each filter 3x3 size
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2, stride=2),


        nn.Conv2d(32,64,kernel_size = 3,padding = 'same'),#input features is basically channel number
        #32 is basically the number of filters, out channel , each filter 3x3 size
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)




    )
    self.classifier = nn.Sequential(
      nn.Flatten(),
      nn.Linear( 64*7*7,128), # Changed from 64*7*6 to 64*7*7 to match actual tensor size of 3136
      nn.ReLU(),
      nn.Dropout(p=0.4),

      nn.Linear(128,64),
      nn.ReLU(),
      nn.Dropout(p=0.4),

      nn.Linear(64,10)





    )
    ##we dont need to explicitly define softmax here as in pytorch in cross entropy loss has softmax implemented internally
  def forward(self,X):
    x = self.features(X)
    x = self.classifier(x)
    return x

In [52]:
# set learning rate and epochs
learning_rate = 0.01
epochs = 100

In [53]:
#instatiate the model
import torch.optim as optim
mynn = MyNN(1)


#loss function
loss_function = nn.CrossEntropyLoss()
#optimizer

optimizer = optim.SGD(mynn.parameters(), lr=learning_rate,weight_decay = 1e-4)


In [54]:
#training loop:
for epoch in range(epochs):
  total_epoch_loss= 0
  for batch_train , batch_label in train_loader:
    # Reshape flattened 1D vectors [784] into [1, 28, 28] channel/spatial format
    batch_train = batch_train.view(-1, 1, 28, 28)

    #forward pass
    y_pred = mynn(batch_train)


    #calculate loss
    loss = loss_function(y_pred ,batch_label)
    #back pass
    optimizer.zero_grad()
    loss.backward()


    #upgrade grades
    optimizer.step()
    total_epoch_loss += loss.item()
    avg_loss = total_epoch_loss / len(train_loader)

  print(f'epoch {epoch+1} : {avg_loss}')

epoch 1 : 1.797052402496338
epoch 2 : 1.0676177406311036
epoch 3 : 0.8567522486050924
epoch 4 : 0.7394230985641479
epoch 5 : 0.6852606880664825
epoch 6 : 0.6285970248778661
epoch 7 : 0.6094285056988398
epoch 8 : 0.5654204827547074
epoch 9 : 0.5334436197082202
epoch 10 : 0.5083559372027715
epoch 11 : 0.4860963984330495
epoch 12 : 0.4805117505788803
epoch 13 : 0.4467060708999634
epoch 14 : 0.4409762542446454
epoch 15 : 0.42220021908481914
epoch 16 : 0.4016233050823212
epoch 17 : 0.37910159533222515
epoch 18 : 0.37822977075974146
epoch 19 : 0.35668066347638766
epoch 20 : 0.3496341640750567
epoch 21 : 0.3348114503423373
epoch 22 : 0.3210569468140602
epoch 23 : 0.30346024786432585
epoch 24 : 0.2949423998097579
epoch 25 : 0.2894636054088672
epoch 26 : 0.2668899965286255
epoch 27 : 0.2643039093414942
epoch 28 : 0.25777430027723314
epoch 29 : 0.25510610108574233
epoch 30 : 0.24074213415384293
epoch 31 : 0.23864262253046037
epoch 32 : 0.2247957181930542
epoch 33 : 0.22452001941700778
epoch 34 :

In [55]:
#set model to eval mode
mynn.eval()
#there are certain features which are different in eval and train, dropout, mean-std calculation

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [56]:
#evaluation code
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:
    batch_features = batch_features.view(-1, 1, 28, 28)
    outputs = mynn(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.8758333333333334


In [57]:
#evaluation code
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in train_loader:
    batch_features = batch_features.view(-1, 1, 28, 28)
    outputs = mynn(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

1.0
